# Phase 1
Training the teacher to avoid obstacles and get to the goal efficiently/quickly. I will likely convert this to python so that I can import it into another file where I run phase 1, then phase 2 and 3, rather than having to run each separately, but I'm not sure yet. 

In [1]:
%cd ..
# need to be able to see environment
# this isn't working for some reason

/Users/charlottewoodrum/Documents/GitHub/gymnasium-rl/gridworld_v1


In [2]:
%pip install torch


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install tqdm
%pip install matplotlib


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
os.listdir()
# os.chdir("GitHub/gymnasium-rl/gridworld_v1")

['.DS_Store',
 'requirements.txt',
 'checkpoint_models',
 'training',
 'all_models',
 'agents',
 'environment']

In [5]:
# imports
from environment.custom_environment import GridWorldBase, TeacherWrapper
from agents.agents import TeacherAgent, ExperienceReplay
import tqdm
import torch
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
import random
import re
import glob
import os

In [6]:
changes = input("What did you change for this round? ")

In [7]:
# initialization
grid_size = 6
learning_rate = 0.001
initial_epsilon = 1
epsilon_decay = 0.9995
final_epsilon = 0.05
discount_factor = 0.99

episodes = 10000
max_steps = 25 # 25 tiles in whole gridworld
target_update_freq = 500 # for target CNN, in steps
visibility = 4

goal_reward = 10
step_penalty = -0.5

num_special_regions = 1
special_region_rewards = [-5.0]

experience_capacity = 8000
batch_size = 64

grid_sizes = [5, 9] # training on different sizes for better generalizability, these grid sizes are arbitrary

num_filters_first_layer = 16
final_conv_filters = num_filters_first_layer * 2
target_spatial_size = 3 # needs to evenly divide visibility * 2 + 1

base_env = GridWorldBase(grid_size, num_special_regions, goal_reward, step_penalty)
env = TeacherWrapper(base_env, visibility, max_steps, special_region_rewards)
agent = TeacherAgent(num_special_regions, learning_rate, initial_epsilon, epsilon_decay, final_epsilon, num_filters_first_layer, final_conv_filters, target_spatial_size, target_update_freq, discount_factor)

experience_replays = {
    g: ExperienceReplay(capacity=experience_capacity, batch_size=batch_size) for g in grid_sizes
}

Using mps device


In [8]:
# training
episode_total_rewards = []
losses = []
lengths = defaultdict(list)

current_phase = 0 # indices of grid_sizes at and before this number will be trained on
episode_markers_for_phase_switch = [] # 3000, 8000

pbar = tqdm.tqdm(range(episodes), desc="Training")

for episode in pbar:
    grid_size = random.choice(grid_sizes)

    max_steps = grid_size * 4 # scaling up
    base_env = GridWorldBase(grid_size, num_special_regions, goal_reward, step_penalty)
    env = TeacherWrapper(base_env, visibility, max_steps, special_region_rewards)
    replay = experience_replays[grid_size]

    episode_losses = [] 
    obs, info = env.reset()
    state = base_env.make_one_agent_grid_relative("teacher", visibility)
    episode_reward = 0

    for step in range(max_steps): 
        action = agent.get_action(state)
        obs, reward, terminated, truncated, info = env.step(action)
        next_state = base_env.make_one_agent_grid_relative("teacher", visibility)

        replay.add_experience(state, action, reward, next_state, terminated or truncated) # used to just be terminated, but added truncated to stop bleeding btw episodes which happens when done isn't triggered
        
        episode_reward += reward
        state = next_state

        if replay.can_provide_sample() and step % 4 == 0: # don't learn every step (improved performance I think)
            experiences = replay.sample_batch()
            loss = agent.learn(experiences)
            episode_losses.append(loss)
            pbar.set_postfix(epsilon=f"{agent.epsilon:.3f}", reward=f"{episode_reward:.1f}", steps=step, loss=f"{loss:.3f}")
        

        if terminated or truncated: 
            lengths[grid_size].append(step + 1) # this is very variable depending on grid size, so I'm storing with the grid size as a key
            break

    avg_loss = sum(episode_losses) / len(episode_losses) if episode_losses else 0
    pbar.set_postfix(epsilon=f"{agent.epsilon:.3f}", reward=f"{episode_reward:.1f}", steps=step + 1, loss=f"{avg_loss:.3f}", refresh=False)

    if episode % 500 == 0 and len(episode_total_rewards) >= 100:
        avg = sum(episode_total_rewards[-100:]) / 100
        print(f"ep {episode} | avg_reward(100)={avg:.1f} | epsilon={agent.epsilon:.3f}", flush=True)
    
    losses.extend(episode_losses)
    agent.decay_epsilon()
    
    episode_total_rewards.append(episode_reward)

pbar.close()

Training:   5%|▍         | 498/10000 [00:32<11:46, 13.44it/s, epsilon=0.779, loss=0.149, reward=-13.0, steps=16]

ep 500 | avg_reward(100)=-20.8 | epsilon=0.779


Training:  10%|▉         | 999/10000 [01:04<07:53, 19.02it/s, epsilon=0.606, loss=0.258, reward=6.0, steps=8]   

ep 1000 | avg_reward(100)=-9.4 | epsilon=0.606


Training:  15%|█▌        | 1500/10000 [01:31<09:07, 15.53it/s, epsilon=0.472, loss=0.677, reward=-11.5, steps=4] 

...

Training:  90%|████████▉ | 8998/10000 [09:35<00:41, 24.19it/s, epsilon=0.050, loss=0.387, reward=6.0, steps=8]    

ep 9000 | avg_reward(100)=0.7 | epsilon=0.050


Training:  95%|█████████▍| 9498/10000 [10:20<00:30, 16.66it/s, epsilon=0.050, loss=0.229, reward=-0.5, steps=0]  

ep 9500 | avg_reward(100)=2.5 | epsilon=0.050


Training: 100%|██████████| 10000/10000 [10:47<00:00, 15.45it/s, epsilon=0.050, loss=0.132, reward=9.5, steps=2]   


In [9]:
agent.epsilon = 0
base_env = GridWorldBase(9, num_special_regions)
env = TeacherWrapper(base_env, visibility, 36, special_region_rewards)
obs, info = env.reset()
state = base_env.make_one_agent_grid_relative("teacher", visibility)

for step in range(36):
    state_tensor = torch.tensor(state, dtype=torch.float32, device=agent.device).unsqueeze(0)
    compass = state_tensor[0, -2:, 0, 0]
    with torch.no_grad():
        q_values = agent.model(state_tensor)
    print(f"step {step} | compass: row={compass[0]:.3f} col={compass[1]:.3f} | q_values: {q_values[0].tolist()} | action: {q_values[0].argmax().item()}")
    
    action = agent.get_action(state)
    obs, reward, terminated, truncated, info = env.step(action)
    state = base_env.make_one_agent_grid_relative("teacher", visibility)
    base_env.render()
    
    if terminated or truncated:
        print(f"done in {step+1} steps")
        break

step 0 | compass: row=0.750 col=-0.750 | q_values: [-2.570526123046875, -6.496562957763672, 6.293283462524414, 6.702282905578613] | action: 3
['  ', '0 ', '  ', '  ', '  ', '  ', '  ', '  ', '  ']
['  ', '  ', '  ', '  ', '0 ', '  ', '0 ', '  ', '  ']
['  ', '  ', '0 ', '🟨 ', '0 ', '0 ', '0 ', '  ', '0 ']
['  ', '  ', '  ', '🟦 ', '0 ', '  ', '  ', '  ', '  ']
['  ', '  ', '  ', '  ', '0 ', '  ', '  ', '  ', '  ']
['🟩 ', '  ', '  ', '  ', '  ', '  ', '  ', '  ', '  ']
['0 ', '0 ', '  ', '  ', '  ', '  ', '0 ', '0 ', '  ']
['  ', '  ', '0 ', '0 ', '0 ', '  ', '0 ', '  ', '0 ']
['0 ', '  ', '  ', '0 ', '0 ', '  ', '  ', '0 ', '  ']

step 1 | compass: row=0.500 col=-0.750 | q_values: [2.3603830337524414, 3.2876014709472656, 8.662858009338379, 7.866608619689941] | action: 2
['  ', '0 ', '  ', '  ', '  ', '  ', '  ', '  ', '  ']
['  ', '  ', '  ', '  ', '0 ', '  ', '0 ', '  ', '  ']
['  ', '  ', '0 ', '🟨 ', '0 ', '0 ', '0 ', '  ', '0 ']
['  ', '  ', '🟦 ', '  ', '0 ', '  ', '  ', '  ', '  ']


In [10]:
# works no matter what random directory it ends up running from
if os.path.exists("../all_models"):
    base_dir = "../all_models"
elif os.path.exists("all_models"):
    base_dir = "all_models"
else:
    raise FileNotFoundError("Could not find all_models")

dirs = glob.glob(os.path.join(base_dir, "tm_*"))
highest_num = 0
for d in dirs: 
    number = re.search(r'tm_(\d+)$', d)
    if number: 
        highest_num = max(highest_num, int(number.group(1)))

next_num = highest_num + 1
run_dir = os.path.join(base_dir, f"tm_{next_num}")
os.makedirs(run_dir, exist_ok=True)

hyperparameters = {
    "grid_size": grid_size, 
    "learning_rate": learning_rate, 
    "initial_epsilon": initial_epsilon, 
    "epsilon_decay": epsilon_decay, 
    "final_epsilon": final_epsilon, 
    "discount_factor": discount_factor, 
    "episodes": episodes, 
    "max_steps": max_steps, 
    "target_update_freq": target_update_freq, 
    "visibility": visibility, 
    "goal_reward": goal_reward, 
    "num_special_regions": num_special_regions, 
    "special_region_rewards": special_region_rewards, 
    "experience_capacity": experience_capacity, 
    "batch_size": batch_size, 
    "grid_sizes": grid_sizes, 
    "num_filters_first_layer": num_filters_first_layer, 
    "final_conv_filters": final_conv_filters, 
    "target_spatial_size": target_spatial_size, 
    "step_penalty": step_penalty
}

checkpoint = {
    "hyperparameters": hyperparameters,
    "model_state_dict": agent.model.state_dict(),
}

torch.save(checkpoint, f'{run_dir}/model_info.pt')

print(f"base_dir: {base_dir}")
print(f"run_dir: {run_dir}")

base_dir: all_models
run_dir: all_models/tm_46


In [11]:
print("cwd:", os.getcwd())
print("dirs:", dirs)
print("base_dir exists:", os.path.exists(base_dir))

cwd: /Users/charlottewoodrum/Documents/GitHub/gymnasium-rl/gridworld_v1
dirs: ['all_models/tm_40', 'all_models/tm_25', 'all_models/tm_22', 'all_models/tm_14', 'all_models/tm_13', 'all_models/tm_41', 'all_models/tm_12', 'all_models/tm_15', 'all_models/tm_23', 'all_models/tm_24', 'all_models/tm_8', 'all_models/tm_1', 'all_models/tm_6', 'all_models/tm_7', 'all_models/tm_9', 'all_models/tm_39', 'all_models/tm_37', 'all_models/tm_30', 'all_models/tm_31', 'all_models/tm_36', 'all_models/tm_38', 'all_models/tm_21', 'all_models/tm_26', 'all_models/tm_10', 'all_models/tm_17', 'all_models/tm_28', 'all_models/tm_43', 'all_models/tm_44', 'all_models/tm_16', 'all_models/tm_29', 'all_models/tm_11', 'all_models/tm_27', 'all_models/tm_18', 'all_models/tm_20', 'all_models/tm_45', 'all_models/tm_42', 'all_models/tm_2', 'all_models/tm_3', 'all_models/tm_4', 'all_models/tm_33', 'all_models/tm_34', 'all_models/tm_35', 'all_models/tm_32']
base_dir exists: True


In [12]:
import glob, os
glob.glob("../all_models/tm_*")

[]

In [13]:
# https://gymnasium.farama.org/introduction/train_agent/
def get_moving_avgs(arr, window, convolution_mode):
    """Compute moving average to smooth noisy data."""
    return np.convolve(
        np.array(arr).flatten(),
        np.ones(window),
        mode=convolution_mode
    ) / window

# Smooth over this window
rolling_length = episodes//20
fig, axs = plt.subplots(ncols=2, figsize=(12, 5))

# Episode rewards (win/loss performance)
axs[0].set_title("Episode rewards")
reward_moving_average = get_moving_avgs(
    episode_total_rewards,
    rolling_length,
    "valid"
)
axs[0].plot(range(len(reward_moving_average)), reward_moving_average)
axs[0].set_ylabel("Average Reward")
axs[0].set_xlabel("Episode")


# Training error (how much we're still learning)
axs[1].set_title("Training Error")
training_error_moving_average = get_moving_avgs(
    losses,
    rolling_length,
    "same"
)
axs[1].plot(range(len(training_error_moving_average)), training_error_moving_average)
axs[1].set_ylabel("Temporal Difference Error")
axs[1].set_xlabel("Step")

plt.tight_layout()
plt.savefig(f'{run_dir}/plots.png')
print(f"Saved to {run_dir}")
plt.close()

Saved to all_models/tm_46


In [14]:
rolling_length = episodes // (len(grid_sizes) * 10) # bc split across diff grid sizes
fig, axs = plt.subplots(ncols=len(grid_sizes), figsize=(12, 5))
axs = np.atleast_1d(axs) # force to always return an array, even if only 1 length

for i, key in enumerate(grid_sizes): 
    # Episode lengths (num steps, to reach goal or to truncation)
    axs[i].set_title(f"Episode lengths: {key}")
    if grid_size not in lengths or len(lengths[key]) < rolling_length: 
        continue # not enough data for grid size
    length_moving_average = get_moving_avgs(
        lengths[key],
        rolling_length,
        "valid"
    )
    axs[i].plot(range(len(length_moving_average)), length_moving_average)
    axs[i].set_ylabel("Average Episode Length")
    axs[i].set_xlabel("Episode")


plt.tight_layout()
plt.savefig(f'{run_dir}/length_plots.png')
print(f"Saved to {run_dir}")
plt.close()

Saved to all_models/tm_46


In [15]:
# load agent
'''
run_dir = "all_models/tm_6"

model_path = os.path.join(run_dir, "model.pt")
checkpoint = torch.load(model_path)

# hyperparameters = checkpoint["hyperparameters"]

agent.model.load_state_dict(checkpoint)
'''

'\nrun_dir = "all_models/tm_6"\n\nmodel_path = os.path.join(run_dir, "model.pt")\ncheckpoint = torch.load(model_path)\n\n# hyperparameters = checkpoint["hyperparameters"]\n\nagent.model.load_state_dict(checkpoint)\n'

In [ ]:
with open(f'{run_dir}/test_result.txt', 'w') as f:
    def run_test(world_size, output_file=None):
        base_env = GridWorldBase(world_size, num_special_regions)
        env = TeacherWrapper(base_env, visibility, max_steps, special_region_rewards)
        obs, info = env.reset()
        total_reward = 0
        state = base_env.make_one_agent_grid_relative("teacher", visibility)
        went_through_special_region = False
        
        for step in range(max_steps):
            action = agent.get_action(state)
            obs, reward, terminated, truncated, info = env.step(action)
            next_state = base_env.make_one_agent_grid_relative("teacher", visibility)
            if reward in special_region_rewards:
                went_through_special_region = True
            total_reward += reward
            print(f"Step {step} | Action: {action} | Reward: {reward}", file=output_file)
            state = next_state
            if terminated or truncated:
                if truncated:
                    print(f"Truncated after {step+1} steps | Total reward: {total_reward}", file=output_file)
                if terminated:
                    print(f"Reached goal after {step+1} steps | Total reward: {total_reward}", file=output_file)
                if went_through_special_region:
                    print("Went through special region", file=output_file)
                break
    
    for label, size in [("Same size", grid_size), ("2x size", grid_size*2)]:
        print(f"\n{label} tests:", file=f)
        for i in range(1, 4):
            print(f"\nTest {i}:\n", file=f)
            run_test(size, output_file=f)

In [17]:
# this runs at the end so that I can see graphs/results before writing my note
human_readable = f""
for param in hyperparameters: 
    human_readable += f"{str(param)} = {hyperparameters[param]}\n"

human_readable_time = pbar.format_interval(pbar.format_dict['elapsed'])
human_readable += f"\n\nTraining Time: {human_readable_time}\n"
if len(changes) > 0: # recorded before training
    human_readable += "Changes: " + changes + "\n"
notes = input("Notes on this training run (consider what won't be recorded, like changes to envs or agents): ")
if len(notes) > 0: 
    human_readable += "Notes: " + notes

with open(f'{run_dir}/human_info.txt', 'w') as f: 
    f.write(human_readable)